# 2. Victim Classification — Bridge vs Real Victim

**Problem**: When tracing incoming transfers to an attacker, we find many senders.
Some are actual victims (drained). Others are bridges, DEX routers, or the attacker's own wallets.

**Goal**: Classify each sender as:
- `real_victim` — A person who lost funds
- `bridge_pass_through` — Bridge contract forwarding funds
- `dex_router` — DEX aggregator/router
- `self_transfer` — Attacker's own wallet
- `dust` — Tiny unsolicited transfer (not a victim)

**Key Features**:
- Transfer amount relative to attacker's total
- Time clustering (1000 victims drained in 2 weeks → same column in timeline)
- Sender's overall behavior pattern
- Known entity matching

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from notebooks.src.data_loader import DataLoader
from notebooks.src.investigation_visualizer import InvestigationVisualizer as iviz
from api.algorithms.wallet_heuristics import classify_from_transfer_stats

loader = DataLoader()
print('Connected')

## 2.1 Load Investigation Data

In [ ]:
# Load the most recent investigation
INVESTIGATION_ID = 1  # Change to your investigation ID

transfers = loader.get_transfers(INVESTIGATION_ID)
wallets = loader.get_wallets(INVESTIGATION_ID)

print(f'Transfers: {len(transfers)}')
print(f'Wallets: {len(wallets)}')
if not wallets.empty:
    display(wallets.groupby('role').size())

## 2.2 Identify Incoming Senders (Potential Victims)

In [ ]:
# Find all attacker wallets
attackers = wallets[wallets['role'].isin(['attacker', 'theft_origin'])]
attacker_addrs = set(attackers['address'].str.lower())
print(f'Attacker addresses: {len(attacker_addrs)}')

# Find all incoming transfers TO attacker addresses
incoming_to_attacker = transfers[transfers['to_address'].str.lower().isin(attacker_addrs)].copy()
print(f'Incoming transfers to attackers: {len(incoming_to_attacker)}')

# Group by sender
sender_stats = incoming_to_attacker.groupby('from_address').agg(
    tx_count=('value', 'count'),
    total_value=('value', 'sum'),
    avg_value=('value', 'mean'),
    first_tx=('timestamp', 'min'),
    last_tx=('timestamp', 'max'),
    unique_tokens=('token_symbol', 'nunique'),
).reset_index()

print(f'Unique senders: {len(sender_stats)}')
display(sender_stats.nlargest(10, 'total_value'))

## 2.3 Classify Senders Using Heuristics

In [ ]:
# Load known entities for matching
known = loader.get_known_entities()
known_addrs = set(known['address'].str.lower()) if not known.empty else set()
known_map = dict(zip(known['address'].str.lower(), known['type'])) if not known.empty else {}

# Classify each sender
classifications = []
for _, sender in sender_stats.iterrows():
    addr = sender['from_address'].lower()
    
    # Known entity?
    if addr in known_map:
        category = f"{known_map[addr]}_pass_through"
        confidence = 1.0
    # Dust filter
    elif sender['total_value'] < 0.001:
        category = 'dust'
        confidence = 0.9
    # Self-transfer (also appears as outgoing from attacker)
    elif addr in attacker_addrs:
        category = 'self_transfer'
        confidence = 1.0
    # Single large transfer = likely real victim
    elif sender['tx_count'] <= 3 and sender['total_value'] > 10:
        category = 'real_victim'
        confidence = 0.7
    # Many small transfers from same sender = bot/service
    elif sender['tx_count'] > 20:
        category = 'bot_or_service'
        confidence = 0.5
    else:
        category = 'unknown'
        confidence = 0.3
    
    classifications.append({
        'address': addr,
        'category': category,
        'confidence': confidence,
        'total_value': sender['total_value'],
        'tx_count': sender['tx_count'],
    })

class_df = pd.DataFrame(classifications)
print(f'Classification results:')
display(class_df.groupby('category').agg(
    count=('address', 'size'),
    total_value=('total_value', 'sum'),
    avg_confidence=('confidence', 'mean'),
).sort_values('total_value', ascending=False))

## 2.4 Time Clustering — Same-Week Victims

In [ ]:
# Group victims by the week they were drained
if not incoming_to_attacker.empty:
    incoming_to_attacker['week'] = incoming_to_attacker['timestamp'].dt.isocalendar().week
    incoming_to_attacker['year_week'] = (
        incoming_to_attacker['timestamp'].dt.year.astype(str) + '-W' + 
        incoming_to_attacker['week'].astype(str).str.zfill(2)
    )
    
    weekly = incoming_to_attacker.groupby('year_week').agg(
        unique_senders=('from_address', 'nunique'),
        total_value=('value', 'sum'),
        tx_count=('value', 'count'),
    ).sort_index()
    
    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Bar(x=weekly.index, y=weekly['unique_senders'], 
                         name='Unique Victims', marker_color='#d62728', opacity=0.7),
                  secondary_y=False)
    fig.add_trace(go.Scatter(x=weekly.index, y=weekly['total_value'], 
                             name='Total Value Drained', line=dict(color='#1f77b4', width=2)),
                  secondary_y=True)
    fig.update_layout(title='Victim Draining Timeline (Weekly)', template='plotly_white', height=400)
    fig.update_yaxes(title_text='# Unique Victims', secondary_y=False)
    fig.update_yaxes(title_text='Total Value', secondary_y=True)
    fig.show()
    
    # Weeks with 10+ victims → mark as mass drain event
    mass_drain_weeks = weekly[weekly['unique_senders'] >= 10]
    if not mass_drain_weeks.empty:
        print(f'\nMass drain events (10+ victims/week):')
        display(mass_drain_weeks)
else:
    print('No incoming transfers to attacker found')

## 2.5 Limitations & Tags

| Limitation | Impact | Mitigation |
|:-----------|:-------|:-----------|
| Bridge detection without bytecode | May miss unknown bridges | Etherscan contract ABI check |
| DEX router detection | Uniswap/Sushi routers look like victims | Maintain known router list |
| Cross-chain victims | Bridge-mediated drains not tracked | Need multi-chain expansion |

**TAG: `aria_bridge_cases`** — When a transfer goes through a bridge, Aria can manually trace the other chain.

**TAG: `threshold_tuning_needed`** — The 0.001 dust threshold and 10-value victim threshold should be data-driven.